In [ ]:
# ============================================================
# COMPLEX MULTIPLICATION WITH QUANTIZATION:
# EXACT IMPLEMENTATION AND PQN NOISE MODEL
# ============================================================
#
# Consider the complex numbers
#
#                   x = alpha + j beta
#
#                   y = gamma + j delta.
#
# Their ideal product is
#
#       xy = (alpha gamma - beta delta)
#            + j(beta gamma + alpha delta)
#
#          = epsilon + j zeta.
#
#
# ============================================================
# REAL-MULTIPLICATION IMPLEMENTATION
# ============================================================
#
# Four real products are required:
#
#       alpha gamma
#       beta delta
#       beta gamma
#       alpha delta
#
# After each real multiplication, finite-precision arithmetic
# introduces a quantization operation.
#
# The real and imaginary parts are then formed as
#
#       epsilon = alpha gamma - beta delta
#
#       zeta    = beta gamma + alpha delta.
#
#
# ============================================================
# FOUR- AND SIX-QUANTIZER MODELS
# ============================================================
#
# FOUR-QUANTIZER MODEL
#
# Quantization occurs only after the four real multiplications:
#
#       Q(alpha gamma)
#       Q(beta delta)
#       Q(beta gamma)
#       Q(alpha delta).
#
# The output is then
#
#       epsilon_q = Q(alpha gamma) - Q(beta delta)
#
#       zeta_q    = Q(beta gamma) + Q(alpha delta).
#
#
# SIX-QUANTIZER MODEL
#
# Two additional quantizers are placed after the final
# subtraction and addition:
#
#       epsilon_q
#
#       = Q[Q(alpha gamma) - Q(beta delta)]
#
#       zeta_q
#
#       = Q[Q(beta gamma) + Q(alpha delta)].
#
#
# ============================================================
# PQN LINEARIZATION
# ============================================================
#
# Quantizers make the system nonlinear.
#
# In the standard PQN approximation, each quantizer is replaced
# by an additive quantization-noise source.
#
# For the real output:
#
#       n_epsilon = eta_ag - eta_bd
#
# in the four-quantizer model, and
#
#       n_epsilon = eta_ag - eta_bd + eta_epsilon
#
# in the six-quantizer model.
#
#
# For the imaginary output:
#
#       n_zeta = eta_bg + eta_ad
#
# in the four-quantizer model, and
#
#       n_zeta = eta_bg + eta_ad + eta_zeta
#
# in the six-quantizer model.
#
#
# ============================================================
# QUANTIZATION-NOISE VARIANCE
# ============================================================
#
# With K fractional bits,
#
#                   Delta = 2^(-K).
#
# Under the standard rounding-noise approximation, every
# individual quantization-noise source is modeled as
#
#                   eta ~ U(-Delta/2, Delta/2)
#
# with
#
#                   E{eta} = 0
#
# and
#
#                   sigma_q^2 = Delta^2 / 12.
#
#
# If the noise sources are mutually uncorrelated:
#
# FOUR QUANTIZERS
#
#       var(n_epsilon) = 2 sigma_q^2
#
#       var(n_zeta)    = 2 sigma_q^2.
#
#
# SIX QUANTIZERS
#
#       var(n_epsilon) = 3 sigma_q^2
#
#       var(n_zeta)    = 3 sigma_q^2.
#
#
# The real and imaginary output-noise components are also
# uncorrelated under these assumptions:
#
#                   E{n_epsilon n_zeta} = 0.
#
#
# ============================================================
# HOW TO USE THIS NOTEBOOK
# ============================================================
#
# 1. Select one of the four display modes.
#
# 2. Only the controls that affect the selected display remain
#    enabled. The others are automatically disabled.
#
#
# COMPLEX PRODUCT
#
#   Change:
#
#       alpha
#       beta
#       gamma
#       delta
#       K
#       number of quantizers
#
#   Compare the ideal complex product with the actual
#   finite-precision result in the complex plane.
#
#
# REAL AND IMAGINARY ERRORS
#
#   Change:
#
#       alpha
#       beta
#       gamma
#       delta
#       K
#       number of quantizers
#
#   Observe separately the quantization error in the real and
#   imaginary parts.
#
#
# OUTPUT-NOISE SCATTER
#
#   Change:
#
#       K
#       number of quantizers
#       Monte Carlo sample count
#
#   Observe the PQN output-noise cloud in the
#
#                   (n_epsilon, n_zeta)
#
#   plane.
#
#
# NOISE VARIANCE VS K
#
#   Change:
#
#       K
#       number of quantizers
#
#   Observe the theoretical dependence of the output-noise
#   variance on the fractional word length.
#
#
# ============================================================
# WHAT WE EXPECT TO OBSERVE
# ============================================================
#
# COMPLEX PRODUCT
#
# Quantization moves the calculated complex product away from
# its ideal position.
#
# Increasing K decreases
#
#                   Delta = 2^(-K)
#
# so the finite-precision result should generally approach the
# ideal complex product.
#
#
# REAL AND IMAGINARY ERRORS
#
# Both output-error components should generally decrease in
# absolute magnitude as K increases.
#
# Their exact values depend on the operands and on the positions
# of the quantization levels.
#
#
# OUTPUT-NOISE SCATTER
#
# Under the PQN assumptions, the noise cloud should be centered
# close to the origin because every individual rounding-noise
# source is zero-mean.
#
# The cloud should not exhibit a strong preferred diagonal
# direction because the real and imaginary output-noise
# components are assumed to be uncorrelated.
#
# Increasing K should make the cloud contract around the origin.
#
# Switching from four to six quantizers introduces one additional
# independent noise source into each output and should therefore
# make the cloud wider.
#
#
# NOISE VARIANCE VS K
#
# Since
#
#                   sigma_q^2 = Delta^2 / 12
#
# and
#
#                   Delta = 2^(-K),
#
# the output-noise variance decreases proportionally to
#
#                   2^(-2K).
#
# Therefore, every additional fractional bit reduces the
# theoretical variance by a factor of four.
#
#
# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================
#
# The first two displays use the ACTUAL nonlinear rounding
# quantizers.
#
# The last two displays use the PQN statistical model.
#
# Thus, the notebook deliberately distinguishes:
#
#       exact finite-precision behavior
#
# from
#
#       statistical linearized noise analysis.
#
# The PQN model is an approximation and relies on the assumption
# that the individual quantization-noise sources are zero-mean
# and mutually uncorrelated.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive
from IPython.display import display


# ------------------------------------------------------------
# Quantizer
# ------------------------------------------------------------

def round_quantizer(x, Delta):

    x = np.asarray(x)

    q = np.where(x >= 0, np.floor(x / Delta + 0.5), np.ceil(x / Delta - 0.5))

    return Delta * q


# ------------------------------------------------------------
# Ideal complex multiplication
# ------------------------------------------------------------

def ideal_complex_product(alpha, beta, gamma, delta):

    epsilon = alpha * gamma - beta * delta

    zeta = beta * gamma + alpha * delta

    return epsilon, zeta


# ------------------------------------------------------------
# Exact finite-precision complex multiplication
# ------------------------------------------------------------

def quantized_complex_product(alpha, beta, gamma, delta, Delta, quantizer_mode):

    ag = alpha * gamma

    bd = beta * delta

    bg = beta * gamma

    ad = alpha * delta

    q_ag = float(round_quantizer(ag, Delta))

    q_bd = float(round_quantizer(bd, Delta))

    q_bg = float(round_quantizer(bg, Delta))

    q_ad = float(round_quantizer(ad, Delta))

    real_before_output_quantizer = q_ag - q_bd

    imag_before_output_quantizer = q_bg + q_ad

    if quantizer_mode == '6 quantizers':

        epsilon_q = float(round_quantizer(real_before_output_quantizer, Delta))

        zeta_q = float(round_quantizer(imag_before_output_quantizer, Delta))

    else:

        epsilon_q = real_before_output_quantizer

        zeta_q = imag_before_output_quantizer

    return epsilon_q, zeta_q, q_ag, q_bd, q_bg, q_ad


# ------------------------------------------------------------
# PQN Monte Carlo simulation
# ------------------------------------------------------------

def simulate_pqn(K, quantizer_mode, N, seed=0):

    rng = np.random.default_rng(seed)

    Delta = 2.0**(-K)

    eta_ag = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

    eta_bd = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

    eta_bg = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

    eta_ad = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

    if quantizer_mode == '6 quantizers':

        eta_epsilon = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

        eta_zeta = rng.uniform(-Delta / 2.0, Delta / 2.0, N)

        noise_real = eta_ag - eta_bd + eta_epsilon

        noise_imag = eta_bg + eta_ad + eta_zeta

        number_of_sources_per_output = 3

    else:

        noise_real = eta_ag - eta_bd

        noise_imag = eta_bg + eta_ad

        number_of_sources_per_output = 2

    sigma_q2 = Delta**2 / 12.0

    theoretical_variance = number_of_sources_per_output * sigma_q2

    return noise_real, noise_imag, Delta, sigma_q2, theoretical_variance


# ------------------------------------------------------------
# Theoretical output-noise variance
# ------------------------------------------------------------

def theoretical_output_variance(K, quantizer_mode):

    Delta = 2.0**(-K)

    sigma_q2 = Delta**2 / 12.0

    if quantizer_mode == '6 quantizers':

        multiplier = 3.0

    else:

        multiplier = 2.0

    return multiplier * sigma_q2


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.cm-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.cm-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.cm-howto {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7cfe0;
    border-left: 6px solid #667da8;
    background: #f8f9fc;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.cm-howto-title {
    font-size: 13px;
    font-weight: bold;
    color: #40587d;
    margin-bottom: 5px;
}

.cm-observe {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7d8c9;
    border-left: 6px solid #3c8a4e;
    background: #f7fbf7;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.cm-observe-title {
    font-size: 13px;
    font-weight: bold;
    color: #245c31;
    margin-bottom: 5px;
}

.cm-model {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.cm-model-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 5px;
}

.cm-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.cm-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.cm-info {
    font-size: 13px;
    line-height: 1.52;
}

.cm-label {
    display: inline-block;
    min-width: 245px;
    font-weight: bold;
}

.cm-value {
    font-size: 14px;
    font-weight: bold;
}

.cm-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 6px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="cm-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Complex Multiplication with Quantization: Exact vs PQN Model
    </div>

</div>
""")


# ------------------------------------------------------------
# Visible introductory documentation
# ------------------------------------------------------------

description_html = HTML("""
<div class="cm-root">

    <div class="cm-description">

        <b>What this notebook demonstrates</b><br><br>

        For

        <div style="text-align:center; margin:6px 0;">
            <b>
            x = α + jβ,
            &nbsp;&nbsp;
            y = γ + jδ,
            </b>
        </div>

        the ideal complex product is

        <div style="text-align:center; margin:6px 0;">
            <b>
            xy =
            (αγ − βδ)
            +
            j(βγ + αδ).
            </b>
        </div>

        A direct implementation requires four real multiplications.
        Quantization after these multiplications makes the implementation
        nonlinear.<br><br>

        The first two displays calculate the <b>actual nonlinear
        finite-precision product</b>. The last two replace the quantizers
        by additive PQN sources and examine the corresponding
        <b>linearized statistical noise model</b>.

    </div>


    <div class="cm-howto">

        <div class="cm-howto-title">
            How to use this notebook
        </div>

        <b>Complex product:</b>
        vary α, β, γ, δ and K and compare the ideal complex result with
        the finite-precision result in the complex plane.<br><br>

        <b>Real and imaginary errors:</b>
        vary the four operands and K and observe separately the errors in
        the real and imaginary components.<br><br>

        <b>Output-noise scatter:</b>
        vary K, the number of quantizers and the Monte Carlo sample count.
        Observe the PQN output-noise cloud in the
        <b>(n<sub>ε</sub>, n<sub>ζ</sub>)</b> plane.<br><br>

        <b>Noise variance vs K:</b>
        vary K and the number of quantizers and observe how the theoretical
        output-noise variance changes with fractional word length.<br><br>

        Controls that do not affect the selected display are automatically
        disabled.

    </div>


    <div class="cm-observe">

        <div class="cm-observe-title">
            What we expect to observe
        </div>

        <b>Complex product:</b>
        quantization moves the calculated complex result away from the
        ideal result. Increasing K decreases
        <b>Δ = 2<sup>−K</sup></b>, so the finite-precision result should
        generally approach the ideal one.<br><br>

        <b>Real and imaginary errors:</b>
        the absolute scale of both error components should generally
        decrease as K increases.<br><br>

        <b>Output-noise scatter:</b>
        under the PQN assumptions, the points should form a cloud centered
        near the origin. There should be no strong preferred diagonal
        direction because the real and imaginary output-noise components
        are assumed to be uncorrelated. Increasing K should contract the
        cloud. Selecting six instead of four quantizers should make it
        wider.<br><br>

        <b>Noise variance vs K:</b>
        the theoretical variance decreases as
        <b>2<sup>−2K</sup></b>. Therefore, every additional fractional
        bit reduces the variance by a factor of four.

    </div>


    <div class="cm-model">

        <div class="cm-model-title">
            Four quantizers versus six quantizers
        </div>

        With <b>four quantizers</b>, rounding is applied only after the
        four real multiplications. Each output therefore contains two
        independent PQN contributions:

        <div style="text-align:center; margin:5px 0;">
            <b>
            σ²<sub>out</sub> = 2Δ²/12.
            </b>
        </div>

        With <b>six quantizers</b>, an additional quantizer is included
        after each final addition/subtraction. Each output then contains
        three independent PQN contributions:

        <div style="text-align:center; margin:5px 0;">
            <b>
            σ²<sub>out</sub> = 3Δ²/12.
            </b>
        </div>

        The PQN model assumes that all involved quantization-noise sources
        are zero-mean and mutually uncorrelated.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(width='610px', min_width='610px', overflow='visible')


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def plot_complex_quantization(display_mode='Complex product', alpha=0.80, beta=0.45, gamma=0.70, delta=-0.35, K=5, quantizer_mode='4 quantizers', N=5000):

    Delta = 2.0**(-K)

    epsilon, zeta = ideal_complex_product(alpha, beta, gamma, delta)

    epsilon_q, zeta_q, q_ag, q_bd, q_bg, q_ad = quantized_complex_product(alpha, beta, gamma, delta, Delta, quantizer_mode)

    error_real = epsilon_q - epsilon

    error_imag = zeta_q - zeta

    complex_error_magnitude = np.sqrt(error_real**2 + error_imag**2)

    noise_real, noise_imag, Delta_pqn, sigma_q2, theoretical_variance = simulate_pqn(K, quantizer_mode, N, seed=0)

    measured_variance_real = np.var(noise_real)

    measured_variance_imag = np.var(noise_imag)

    measured_mean_real = np.mean(noise_real)

    measured_mean_imag = np.mean(noise_imag)

    if np.std(noise_real) > 0.0 and np.std(noise_imag) > 0.0:

        measured_correlation = np.corrcoef(noise_real, noise_imag)[0, 1]

    else:

        measured_correlation = 0.0


    # --------------------------------------------------------
    # Dynamic information
    # --------------------------------------------------------

    if display_mode in ['Complex product', 'Real and imaginary errors']:

        mode_info = f"""
        <span class="cm-label">Ideal real part</span>
        ε = {epsilon:+.8f}
        <br>

        <span class="cm-label">Quantized real part</span>
        ε<sub>q</sub> = {epsilon_q:+.8f}
        <br>

        <span class="cm-label">Real-part error</span>
        {error_real:+.8e}
        <br><br>

        <span class="cm-label">Ideal imaginary part</span>
        ζ = {zeta:+.8f}
        <br>

        <span class="cm-label">Quantized imaginary part</span>
        ζ<sub>q</sub> = {zeta_q:+.8f}
        <br>

        <span class="cm-label">Imaginary-part error</span>
        {error_imag:+.8e}
        <br>

        <span class="cm-label">Complex-error magnitude</span>
        |e| = {complex_error_magnitude:.8e}
        """

    else:

        mode_info = f"""
        <span class="cm-label">Variance of one PQN source</span>
        σ²<sub>q</sub> = Δ²/12 = {sigma_q2:.8e}
        <br>

        <span class="cm-label">Theoretical output variance</span>
        {theoretical_variance:.8e}
        <br>

        <span class="cm-label">Measured real-noise variance</span>
        {measured_variance_real:.8e}
        <br>

        <span class="cm-label">Measured imag-noise variance</span>
        {measured_variance_imag:.8e}
        <br>

        <span class="cm-label">Measured real-noise mean</span>
        {measured_mean_real:+.3e}
        <br>

        <span class="cm-label">Measured imag-noise mean</span>
        {measured_mean_imag:+.3e}
        <br>

        <span class="cm-label">Measured Re/Im correlation</span>
        {measured_correlation:+.5f}
        """


    summary_html.value = f"""
    <div class="cm-box">

        <div class="cm-title">
            Current Quantization Data
        </div>

        <div class="cm-info">

            <span class="cm-label">Selected display</span>
            {display_mode}
            <br>

            <span class="cm-label">Quantizer model</span>
            <span class="cm-value">{quantizer_mode}</span>
            <br>

            <span class="cm-label">Fractional bits</span>
            K = <span class="cm-value">{K}</span>
            <br>

            <span class="cm-label">Quantization step</span>
            Δ = {Delta:.8f}
            <br><br>

            {mode_info}

        </div>

        <div class="cm-note">
            Controls that do not affect the currently selected representation
            are automatically disabled.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(11.8, 5.0))


    # ========================================================
    # COMPLEX PRODUCT
    # ========================================================

    if display_mode == 'Complex product':

        maximum_coordinate = max(abs(epsilon), abs(zeta), abs(epsilon_q), abs(zeta_q), Delta)

        limit = 1.30 * maximum_coordinate

        if limit < 0.25:

            limit = 0.25

        ax.axhline(0.0, linewidth=0.8)

        ax.axvline(0.0, linewidth=0.8)

        ax.plot([0.0, epsilon], [0.0, zeta], linewidth=1.6, linestyle='--', label='Ideal complex product')

        ax.plot([0.0, epsilon_q], [0.0, zeta_q], linewidth=1.8, label='Quantized complex product')

        ax.plot(epsilon, zeta, 'o', markersize=9, label='Ideal result')

        ax.plot(epsilon_q, zeta_q, 's', markersize=8, label='Quantized result')

        ax.plot([epsilon, epsilon_q], [zeta, zeta_q], linestyle=':', linewidth=1.5, label='Quantization displacement')

        ax.set_xlim(-limit, limit)

        ax.set_ylim(-limit, limit)

        ax.set_aspect('equal', adjustable='box')

        ax.set_xlabel('Real part')

        ax.set_ylabel('Imaginary part')

        ax.set_title('Ideal and Quantized Complex Product', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False, fontsize=9)

        plt.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.25)


    # ========================================================
    # REAL AND IMAGINARY ERRORS
    # ========================================================

    elif display_mode == 'Real and imaginary errors':

        positions = np.arange(2)

        values = [error_real, error_imag]

        ax.bar(positions, values, width=0.52)

        ax.axhline(0.0, linewidth=0.8)

        ax.set_xticks(positions)

        ax.set_xticklabels(['Real-part error', 'Imaginary-part error'])

        maximum_error = max(abs(error_real), abs(error_imag), Delta / 2.0)

        ax.set_ylim(-1.30 * maximum_error, 1.30 * maximum_error)

        for position, value in zip(positions, values):

            vertical_alignment = 'bottom' if value >= 0 else 'top'

            offset = 0.05 * maximum_error if value >= 0 else -0.05 * maximum_error

            ax.text(position, value + offset, f'{value:+.4e}', ha='center', va=vertical_alignment, fontsize=10)

        ax.set_ylabel('Actual output error')

        ax.set_title('Real and Imaginary Errors of the Complex Product', fontsize=12)

        ax.grid(True, axis='y', linestyle=':', alpha=0.5)

        plt.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.18)


    # ========================================================
    # OUTPUT-NOISE SCATTER
    # ========================================================

    elif display_mode == 'Output-noise scatter':

        ax.scatter(noise_real, noise_imag, s=9, alpha=0.35)

        ax.axhline(0.0, linewidth=0.8)

        ax.axvline(0.0, linewidth=0.8)

        scatter_limit = 1.15 * max(np.max(np.abs(noise_real)), np.max(np.abs(noise_imag)))

        if scatter_limit == 0.0:

            scatter_limit = Delta

        ax.set_xlim(-scatter_limit, scatter_limit)

        ax.set_ylim(-scatter_limit, scatter_limit)

        ax.set_aspect('equal', adjustable='box')

        ax.set_xlabel(r'Real output noise $n_\epsilon$')

        ax.set_ylabel(r'Imaginary output noise $n_\zeta$')

        ax.set_title(f'PQN Output-Noise Scatter — {quantizer_mode}', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)


        statistics_text = (
            f"Theory variance = {theoretical_variance:.3e}\n"
            f"Measured var(Re) = {measured_variance_real:.3e}\n"
            f"Measured var(Im) = {measured_variance_imag:.3e}\n"
            f"Correlation = {measured_correlation:+.4f}"
        )


        # ----------------------------------------------------
        # Statistical summary placed OUTSIDE the axes
        # ----------------------------------------------------

        ax.text(
            1.05,
            0.97,
            statistics_text,
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.90)
        )


        # Leave free space on the right for the external box

        plt.subplots_adjust(left=0.09, right=0.72, top=0.90, bottom=0.18)


    # ========================================================
    # NOISE VARIANCE VS K
    # ========================================================

    else:

        K_values = np.arange(2, 17)

        variances = np.array([theoretical_output_variance(k, quantizer_mode) for k in K_values])

        current_variance = theoretical_output_variance(K, quantizer_mode)

        ax.plot(K_values, variances, 'o-', linewidth=1.8, markersize=5, label=f'Theoretical variance — {quantizer_mode}')

        ax.plot(K, current_variance, 's', markersize=9, label='Current K')

        ax.set_yscale('log')

        ax.set_xticks(K_values)

        ax.set_xlabel('Fractional bits K')

        ax.set_ylabel('Output-noise variance')

        ax.set_title('PQN Output-Noise Variance versus Word Length', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False, fontsize=9)

        plt.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.25)


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='295px')

slider_style = {'description_width': '105px'}


display_selector = RadioButtons(
    options=[
        'Complex product',
        'Real and imaginary errors',
        'Output-noise scatter',
        'Noise variance vs K'
    ],
    value='Complex product',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(width='300px')
)


quantizer_selector = RadioButtons(
    options=[
        '4 quantizers',
        '6 quantizers'
    ],
    value='4 quantizers',
    description='Model:',
    style={'description_width': '65px'},
    layout=Layout(width='300px')
)


alpha_slider = FloatSlider(
    value=0.80,
    min=-1.50,
    max=1.50,
    step=0.05,
    description='alpha:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


beta_slider = FloatSlider(
    value=0.45,
    min=-1.50,
    max=1.50,
    step=0.05,
    description='beta:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


gamma_slider = FloatSlider(
    value=0.70,
    min=-1.50,
    max=1.50,
    step=0.05,
    description='gamma:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


delta_slider = FloatSlider(
    value=-0.35,
    min=-1.50,
    max=1.50,
    step=0.05,
    description='delta:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


K_slider = IntSlider(
    value=5,
    min=2,
    max=16,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


N_slider = IntSlider(
    value=5000,
    min=1000,
    max=20000,
    step=1000,
    description='MC samples:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


# ------------------------------------------------------------
# Enable only controls relevant to selected display
# ------------------------------------------------------------

def update_control_states(change=None):

    mode = display_selector.value

    if mode == 'Complex product':

        alpha_slider.disabled = False
        beta_slider.disabled = False
        gamma_slider.disabled = False
        delta_slider.disabled = False
        K_slider.disabled = False
        quantizer_selector.disabled = False
        N_slider.disabled = True

    elif mode == 'Real and imaginary errors':

        alpha_slider.disabled = False
        beta_slider.disabled = False
        gamma_slider.disabled = False
        delta_slider.disabled = False
        K_slider.disabled = False
        quantizer_selector.disabled = False
        N_slider.disabled = True

    elif mode == 'Output-noise scatter':

        alpha_slider.disabled = True
        beta_slider.disabled = True
        gamma_slider.disabled = True
        delta_slider.disabled = True
        K_slider.disabled = False
        quantizer_selector.disabled = False
        N_slider.disabled = False

    else:

        alpha_slider.disabled = True
        beta_slider.disabled = True
        gamma_slider.disabled = True
        delta_slider.disabled = True
        K_slider.disabled = False
        quantizer_selector.disabled = False
        N_slider.disabled = True


display_selector.observe(update_control_states, names='value')


# ------------------------------------------------------------
# Initial control state
# ------------------------------------------------------------

update_control_states()


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_complex_quantization,
    display_mode=display_selector,
    alpha=alpha_slider,
    beta=beta_slider,
    gamma=gamma_slider,
    delta=delta_slider,
    K=K_slider,
    quantizer_mode=quantizer_selector,
    N=N_slider
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='cm-title'>Controls</div>"),
        display_selector,
        quantizer_selector,
        alpha_slider,
        beta_slider,
        gamma_slider,
        delta_slider,
        K_slider,
        N_slider
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(width='auto', overflow='visible')


# ------------------------------------------------------------
# Final layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)